# Analisis eksperimen SAUCIS

**JavaScript vs Rust→WebAssembly, main thread vs Web Worker, dan siklus hidup worker** (regresi logistik biner IRLS, aplikasi Statify).

Notebook ini menjalankan ulang seluruh kode Python dari data mentah run `20261020-070557`: 20 sesi ukur (10 Chrome 155, 10 Firefox 158), 28.800 run, ditambah 10 invocation baseline native. Hasilnya adalah semua angka, CSV, gambar, dan tabel di naskah.

| Bagian | Isi | Hasil |
|---|---|---|
| 1. Persiapan | konfigurasi, fungsi bantu, unggah dan ekstrak zip hasil | `data/run-20261020-070557/` |
| 2. Kode analisis | 6 skrip lewat `%%writefile` | `analysis/*.py` |
| 3. Analisis | `analyze.py`, `responsiveness.py`, `extra.py` | `out/*.csv`, `out/numbers*.md` |
| 4. Gambar dan tabel | `make_figures.py`, `make_tables.py` | `figures/` (PDF + PNG), `tables/` |
| 5. Unduh | zip semua hasil | `hasil_analisis_v2.zip` |

**Cara pakai:** isi sel konfigurasi di Bagian 1, lalu pilih **Runtime → Run all**. Notebook meminta Anda mengunggah `20261020-070557.zip` (zip hasil eksperimen yang sama dengan yang dianalisis di naskah). Seluruh notebook selesai dalam sekitar satu menit.

**Metode singkat** (sama dengan naskah):
- Unit replikasi adalah sesi (satu proses browser baru). Dalam satu sesi, tiap kombinasi (workload, N, sel) punya 30 run yang diringkas dengan median.
- Interval kepercayaan 95% memakai distribusi t antar 10 sesi per browser. Rasio adalah rata-rata geometrik dari rasio median per sesi.
- Kontras antar browser memakai Welch pada skala log. Biaya per baris per iterasi memakai OLS terhadap N × iterasi.
- Sel: **A** JS main thread, **B-warm/B-cold** JS di worker persistent/per-run, **C** Wasm main thread, **D-warm/D-cold** Wasm di worker persistent/per-run (D-cold = konfigurasi produksi Statify).

**Verifikasi:** notebook ini sudah dijalankan penuh dengan Python 3.12, pandas 2.2.2, numpy 2.0.2, scipy 1.14.1, dan matplotlib 3.10.0 (setara Colab). Hasilnya dibandingkan dengan paket naskah (dibuat dengan pandas 3.0.5): `out/numbers*.md` dan `tables/*.tex` identik byte-per-byte; nilai di `out/*.csv` sama sampai selisih pembulatan floating-point (relatif < 10⁻⁹).

## 1. Persiapan

### 1.1 Konfigurasi
Biarkan `ZIP_HASIL = None` untuk mengunggah zip lewat dialog. Kalau zip sudah ada di Google Drive, isi path-nya; Drive akan di-mount otomatis.

In [ ]:
# ==== KONFIGURASI ====
RUN_ID = "20261020-070557"
WORKDIR = "/content/paper_v2"      # folder kerja di runtime Colab

# Zip hasil eksperimen (berisi folder 20261020-070557/).
#   None -> diminta unggah
#   path -> dibaca langsung, misalnya "/content/drive/MyDrive/statify/20261020-070557.zip"
ZIP_HASIL = None

# Salin zip hasil akhir ke Google Drive juga? None = tidak.
SALIN_KE_DRIVE = None              # misalnya "/content/drive/MyDrive/statify/hasil_v2"

### 1.2 Lingkungan dan fungsi bantu
Colab sudah memuat pandas, numpy, scipy, dan matplotlib. `jalankan()` menjalankan skrip sebagai proses terpisah (sama seperti `python3 analyze.py` di terminal) dan menghentikan **Run all** bila skrip gagal, supaya sel berikutnya tidak memakai hasil lama.

In [ ]:
import glob, hashlib, os, shutil, subprocess, sys, zipfile

DI_COLAB = "google.colab" in sys.modules
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
for d in ["analysis", "data", "out", "figures", "tables"]:
    os.makedirs(d, exist_ok=True)

try:
    import matplotlib, numpy, pandas, scipy
except ImportError:  # di luar Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "scipy", "matplotlib"], check=True)
    import matplotlib, numpy, pandas, scipy

import pandas as pd
from IPython.display import Image, Markdown, display

if DI_COLAB and any(str(p).startswith("/content/drive") for p in (ZIP_HASIL, SALIN_KE_DRIVE) if p):
    from google.colab import drive
    drive.mount("/content/drive")


def jalankan(cmd, cwd=None, batas=8000):
    """Jalankan perintah, tampilkan keluarannya, dan hentikan notebook bila gagal."""
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, capture_output=True, text=True)
    keluaran = ((r.stdout or "") + (r.stderr or "")).rstrip()
    if len(keluaran) > batas:
        keluaran = "... (dipotong)\n" + keluaran[-batas:]
    if keluaran:
        print(keluaran)
    if r.returncode != 0:
        raise RuntimeError(f"Perintah gagal (kode {r.returncode}): {cmd}")
    return r


def ambil_zip(path, keterangan):
    """Kembalikan path zip dari konfigurasi, atau minta diunggah di Colab."""
    if path:
        if not os.path.exists(path):
            raise FileNotFoundError(f"{keterangan}: {path} tidak ditemukan")
        return path
    if not DI_COLAB:
        raise RuntimeError(f"Isi path untuk {keterangan} di sel konfigurasi.")
    from google.colab import files
    print(f"Pilih {keterangan} ...")
    unggahan = files.upload()
    if not unggahan:
        raise RuntimeError("Tidak ada berkas yang diunggah.")
    return os.path.abspath(next(iter(unggahan)))


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


print("Python", sys.version.split()[0])
for m in (numpy, pandas, scipy, matplotlib):
    print(f"  {m.__name__:<11}{m.__version__}")
print("Folder kerja:", os.getcwd(), "| Colab:", DI_COLAB)

### 1.3 Unggah dan ekstrak zip hasil eksperimen
Isi zip disalin ke `data/run-20261020-070557/`, struktur yang dibaca `common.py`. Matriks desain `design/*.f64` (sekitar 17 MB) dilewati karena analisis tidak memakainya. SHA-256 zip dibandingkan dengan zip yang dianalisis di naskah.

In [ ]:
ZIP_HASIL_SHA256 = "276da919c244e5bfb842f3c0488ab22506a92db5d9b73e9be187d10c18109d0d"

zip_hasil = ambil_zip(ZIP_HASIL, "zip hasil eksperimen (20261020-070557.zip)")
h = sha256(zip_hasil)
print("SHA-256 zip:", h)
print("  cocok dengan zip naskah" if h == ZIP_HASIL_SHA256 else "  PERINGATAN: berbeda dari zip naskah, angka bisa berbeda")

RUN_DIR = os.path.join("data", f"run-{RUN_ID}")
shutil.rmtree(RUN_DIR, ignore_errors=True)
with zipfile.ZipFile(zip_hasil) as z:
    nama = [n for n in z.namelist() if not n.endswith("/")]
    akar = sorted((n[: -len("run.json")] for n in nama if os.path.basename(n) == "run.json"), key=len)
    if not akar:
        raise ValueError("run.json tidak ada di zip. Apakah ini zip hasil run?")
    akar = akar[0]
    n_ok = n_lewat = 0
    for n in nama:
        if not n.startswith(akar):
            continue
        rel = n[len(akar):]
        if rel.startswith("design/") and rel.endswith(".f64"):
            n_lewat += 1
            continue
        tujuan = os.path.join(RUN_DIR, rel)
        os.makedirs(os.path.dirname(tujuan), exist_ok=True)
        with z.open(n) as sumber, open(tujuan, "wb") as keluar:
            shutil.copyfileobj(sumber, keluar)
        n_ok += 1
print(f"{n_ok} berkas diekstrak ke {RUN_DIR} ({n_lewat} berkas .f64 dilewati)")

baris = []
for b in ["chrome", "firefox"]:
    for s in sorted(glob.glob(f"{RUN_DIR}/{b}/*/session.json")):
        folder = os.path.dirname(s)
        baris.append((b, os.path.basename(folder).split("-")[0], len(glob.glob(f"{folder}/batch-*.csv"))))
ringkas = pd.DataFrame(baris, columns=["browser", "peran", "csv"])
display(ringkas.groupby(["browser", "peran"]).agg(sesi=("csv", "size"), csv=("csv", "sum")))
print("native:", len(glob.glob(f"{RUN_DIR}/native/invocation-*.csv")), "invocation")
validasi = open(f"{RUN_DIR}/validation.md", encoding="utf-8").read()
print("validation.md ->", next((l for l in validasi.splitlines() if l.startswith("Hasil")), "(baris Hasil tidak ditemukan)"))

## 2. Kode analisis
Sel-sel berikut menulis skrip ke `analysis/`. Isinya sama persis dengan skrip di paket naskah, jadi skrip yang sama juga bisa dijalankan di luar Colab (lihat `analysis/README.md` di paket). Kalau Anda mengubah kode di sini, jalankan ulang sel `%%writefile`-nya sebelum Bagian 3.

### `common.py`
Pemuat data dan fungsi bersama: membaca semua `batch-*.csv` (hanya sesi *measure*), menghitung `startupMs` dan `responseMs`, median per sesi, t-CI antar sesi, dan rasio geometrik.

In [ ]:
%%writefile analysis/common.py
"""Shared loader and helpers for the v2 experiment analysis.

Unit of replication = session (one fresh browser process). Within a session,
each (workload, N, cell) has 30 recorded runs; we summarise them by the median
and then make inferences across the 10 measured sessions per browser
(Georges et al. 2007; Kalibera & Jones 2013).
"""
from __future__ import annotations

import glob
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RUN = os.path.join(ROOT, "data", "run-20261020-070557")
OUT = os.path.join(ROOT, "out")
FIG = os.path.join(ROOT, "figures")
os.makedirs(OUT, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

CELLS = ["A", "B-warm", "B-cold", "C", "D-warm", "D-cold"]
BROWSERS = ["chrome", "firefox"]
NS = [1000, 10000, 50000, 100000]


def load_runs(include_pilot: bool = False) -> pd.DataFrame:
    frames = []
    for f in sorted(glob.glob(os.path.join(RUN, "*", "*", "batch-*.csv"))):
        if f.endswith("-meta.json"):
            continue
        df = pd.read_csv(f, low_memory=False)
        frames.append(df)
    d = pd.concat(frames, ignore_index=True)
    if not include_pilot:
        d = d[d["sessionRole"] == "measure"].copy()
    d["browser"] = d["browserName"]
    # user-facing response time: dispatch -> result on main thread (core) or
    # -> output tables formatted (full); per-run workers add their start-up,
    # which happens before dispatch and is measured separately.
    start = d[["spawnMs", "initMs", "engineInitMs"]].fillna(0).sum(axis=1)
    cold = d["cell"].isin(["B-cold", "D-cold"])
    base = np.where(d["workload"] == "full", d["totalMs"], d["e2eMs"])
    d["startupMs"] = np.where(cold, start, 0.0)
    d["responseMs"] = base + d["startupMs"]
    return d


def load_native() -> pd.DataFrame:
    frames = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(RUN, "native", "invocation-*.csv")))]
    return pd.concat(frames, ignore_index=True)


def session_medians(d: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    keys = ["browser", "sessionId", "sessionIndex", "workload", "N", "cell"]
    return d.groupby(keys, as_index=False)[cols].median()


def t_ci(x, level=0.95):
    """Mean and t-based CI across sessions."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    m = x.mean()
    if n < 2:
        return m, np.nan, np.nan, n
    se = x.std(ddof=1) / np.sqrt(n)
    h = stats.t.ppf(0.5 + level / 2, n - 1) * se
    return m, m - h, m + h, n


def ratio_ci(log_ratios, level=0.95):
    """Geometric-mean ratio and CI from per-session log ratios."""
    m, lo, hi, n = t_ci(log_ratios, level)
    return np.exp(m), np.exp(lo), np.exp(hi), n


def fmt_ms(x):
    return f"{x:.1f}" if abs(x) >= 1 else f"{x:.2f}"

### `analyze.py`
Analisis utama: waktu respons per sel, baseline native, rasio efek, dekomposisi faktorial, kontras browser, biaya per baris per iterasi, start-up worker per-run, drift/baterai, memori Wasm, dan akurasi. Menulis CSV ke `out/` dan semua angka naskah ke `out/numbers.md`.

In [ ]:
%%writefile analysis/analyze.py
"""Main analysis for the v2 experiment. Writes CSV tables to out/ and a
Markdown digest of every number used in the manuscript to out/numbers.md."""
import numpy as np
import pandas as pd

from common import (BROWSERS, CELLS, NS, OUT, load_native, load_runs, ratio_ci,
                    session_medians, t_ci)

d = load_runs()
nat = load_native()
lines = []


def h(s):
    lines.append("\n## " + s + "\n")


def p(s=""):
    lines.append(s)


METRICS = ["computeMs", "e2eMs", "overheadMs", "responseMs", "startupMs", "wasmCoreMs",
           "formatMs", "totalMs", "spawnMs", "initMs", "engineInitMs", "maxFrameGapMs",
           "droppedFrames", "longTaskCount", "longTaskTotalMs", "wasmMemoryMB",
           "spawnMaxFrameGapMs", "spawnDroppedFrames", "dispatchLatencyMs", "returnLatencyMs",
           "workerHandlerMs", "t0PhaseMs", "t1PhaseMs", "nextFrameAfterT1Ms", "wasmSerializeMs"]
sm = session_medians(d, METRICS)
sm.to_csv(f"{OUT}/session_medians.csv", index=False)

# ---------------------------------------------------------------- overview
h("Overview")
p(f"Recorded runs (measure sessions): {len(d):,}; errors {d['error'].notna().sum()}, "
  f"hidden {int(d['pageHidden'].sum())}, clock-check failures {int(d['clockCheckFailed'].sum())}")
p(f"Sessions per browser: {d.groupby('browser')['sessionId'].nunique().to_dict()}")
p(f"Runs per (browser, workload, N, cell): {d.groupby(['browser','workload','N','cell']).size().unique()}")
it = d.groupby("N")[["iterations", "stepHalvings"]].agg(["min", "max"])
p("Iterations by N: " + "; ".join(f"N={n}: {int(it.loc[n, ('iterations','min')])} it, "
                                 f"{int(it.loc[n, ('stepHalvings','min')])} halving" for n in NS))

# ---------------------------------------------------------------- response time table
h("Response time (mean of session medians, ms, 95% t-CI over 10 sessions)")
rows = []
for (b, w, n, c), g in sm.groupby(["browser", "workload", "N", "cell"]):
    for metric in ["responseMs", "computeMs", "e2eMs"]:
        m, lo, hi, k = t_ci(g[metric])
        rows.append(dict(browser=b, workload=w, N=n, cell=c, metric=metric, mean=m, lo=lo, hi=hi, k=k))
rt = pd.DataFrame(rows)
rt.to_csv(f"{OUT}/response_time.csv", index=False)
for b in BROWSERS:
    for w in ["core", "full"]:
        p(f"\n**{b} / {w}** (responseMs)")
        p("| N | " + " | ".join(CELLS) + " |")
        p("|---|" + "---|" * len(CELLS))
        for n in NS:
            vals = []
            for c in CELLS:
                r = rt.query("browser==@b and workload==@w and N==@n and cell==@c and metric=='responseMs'").iloc[0]
                vals.append(f"{r['mean']:.1f} ({r['lo']:.1f}–{r['hi']:.1f})")
            p(f"| {n:,} | " + " | ".join(vals) + " |")

# ---------------------------------------------------------------- native
h("Native reference (IRLS core, warm)")
nmed = nat.groupby(["invocation", "N"], as_index=False)["computeMs"].median()
nrows = []
for n, g in nmed.groupby("N"):
    m, lo, hi, k = t_ci(g["computeMs"])
    nrows.append(dict(N=n, mean=m, lo=lo, hi=hi, k=k))
nt = pd.DataFrame(nrows)
nt.to_csv(f"{OUT}/native.csv", index=False)
for r in nt.itertuples():
    p(f"N={r.N:,}: native computeMs {r.mean:.2f} ({r.lo:.2f}–{r.hi:.2f}), {r.k} invocations")
# ratios to native (computeMs, and Wasm-internal core time)
p("\nRatio of browser computeMs (mean of session medians) to native mean:")
for b in BROWSERS:
    for n in NS:
        nv = nt.query("N==@n")["mean"].iloc[0]
        parts = []
        for c in ["A", "B-warm", "C", "D-warm"]:
            v = rt.query("browser==@b and workload=='core' and N==@n and cell==@c and metric=='computeMs'")["mean"].iloc[0]
            parts.append(f"{c} {v / nv:.2f}×")
        wc = sm.query("browser==@b and workload=='core' and N==@n and cell=='D-warm'")["wasmCoreMs"].mean()
        parts.append(f"D-warm wasmCore {wc / nv:.2f}×")
        p(f"- {b} N={n:,}: " + ", ".join(parts))
# native accuracy vs browser
cols_b = [c for c in nat.columns if c.startswith("beta_")]
br = d.query("workload=='core' and cell=='D-warm'")
mx = 0.0
for n in NS:
    nb = nat.query("N==@n")[cols_b].iloc[0].values
    bb = br.query("N==@n")[[c.replace("beta_", "B_") for c in cols_b]].values
    mx = max(mx, np.max(np.abs(bb - nb)))
p(f"\nMax |β_native − β_D-warm| over N and sessions: {mx:.2e}")

# ---------------------------------------------------------------- effects
h("Effects as ratios of session medians (geometric mean, 95% CI over sessions)")
piv = sm.pivot_table(index=["browser", "sessionId", "workload", "N"], columns="cell", values="responseMs").reset_index()
pivc = sm.pivot_table(index=["browser", "sessionId", "workload", "N"], columns="cell", values="computeMs").reset_index()
contrasts = {
    "Wasm vs JS, main (C/A)": ("C", "A"),
    "Wasm vs JS, worker warm (D-warm/B-warm)": ("D-warm", "B-warm"),
    "Wasm vs JS, worker cold (D-cold/B-cold)": ("D-cold", "B-cold"),
    "Worker vs main, JS warm (B-warm/A)": ("B-warm", "A"),
    "Worker vs main, Wasm warm (D-warm/C)": ("D-warm", "C"),
    "Cold vs warm, JS (B-cold/B-warm)": ("B-cold", "B-warm"),
    "Cold vs warm, Wasm (D-cold/D-warm)": ("D-cold", "D-warm"),
    "Production D-cold vs A": ("D-cold", "A"),
}
erows = []
for name, (num, den) in contrasts.items():
    for src, tab in [("responseMs", piv), ("computeMs", pivc)]:
        for (b, w, n), g in tab.groupby(["browser", "workload", "N"]):
            lr = np.log(g[num] / g[den])
            r, lo, hi, k = ratio_ci(lr)
            erows.append(dict(contrast=name, metric=src, browser=b, workload=w, N=n, ratio=r, lo=lo, hi=hi, k=k))
eff = pd.DataFrame(erows)
eff.to_csv(f"{OUT}/effects.csv", index=False)
for name in contrasts:
    p(f"\n**{name}** (responseMs; computeMs in brackets)")
    for b in BROWSERS:
        for w in ["core", "full"]:
            parts = []
            for n in NS:
                r = eff.query("contrast==@name and metric=='responseMs' and browser==@b and workload==@w and N==@n").iloc[0]
                rc = eff.query("contrast==@name and metric=='computeMs' and browser==@b and workload==@w and N==@n").iloc[0]
                parts.append(f"{n//1000 if n>=1000 else n}k: {r.ratio:.2f} ({r.lo:.2f}–{r.hi:.2f}) [{rc.ratio:.2f}]")
            p(f"- {b}/{w}: " + "; ".join(parts))

# factorial (warm cells): per session log-effects, t-CI
h("Factorial decomposition on ln(responseMs), warm cells, per session then t-CI")
frows = []
for (b, s, w, n), g in piv.groupby(["browser", "sessionId", "workload", "N"]):
    A, B, C, D = [np.log(g[c].iloc[0]) for c in ["A", "B-warm", "C", "D-warm"]]
    frows.append(dict(browser=b, sessionId=s, workload=w, N=n,
                      tech_main=C - A, loc_js=B - A, interaction=(D - C) - (B - A),
                      tech_avg=((C - A) + (D - B)) / 2, loc_avg=((B - A) + (D - C)) / 2))
fac = pd.DataFrame(frows)
facs = []
for (b, w, n), g in fac.groupby(["browser", "workload", "N"]):
    rec = dict(browser=b, workload=w, N=n)
    for col in ["tech_avg", "loc_avg", "interaction"]:
        m, lo, hi, k = t_ci(g[col])
        rec[col] = (np.exp(m) - 1) * 100
        rec[col + "_lo"] = (np.exp(lo) - 1) * 100
        rec[col + "_hi"] = (np.exp(hi) - 1) * 100
    facs.append(rec)
fac_s = pd.DataFrame(facs)
fac_s.to_csv(f"{OUT}/factorial.csv", index=False)
for r in fac_s.itertuples():
    p(f"- {r.browser}/{r.workload} N={r.N:,}: Wasm {r.tech_avg:+.1f}% ({r.tech_avg_lo:+.1f} to {r.tech_avg_hi:+.1f}), "
      f"worker {r.loc_avg:+.1f}% ({r.loc_avg_lo:+.1f} to {r.loc_avg_hi:+.1f}), "
      f"interaction {r.interaction:+.1f}% ({r.interaction_lo:+.1f} to {r.interaction_hi:+.1f})")

# browser contrast for the same cell (Firefox/Chrome), sessions are independent between browsers
h("Browser contrast Firefox/Chrome, ratio of means of session medians (Welch CI on logs)")
from scipy import stats as st
brows = []
for (w, n, c), g in sm.groupby(["workload", "N", "cell"]):
    a = np.log(g.query("browser=='chrome'")["responseMs"].values)
    f = np.log(g.query("browser=='firefox'")["responseMs"].values)
    diff = f.mean() - a.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + f.var(ddof=1) / len(f))
    dfw = se**4 / ((a.var(ddof=1) / len(a))**2 / (len(a) - 1) + (f.var(ddof=1) / len(f))**2 / (len(f) - 1))
    q = st.t.ppf(0.975, dfw)
    brows.append(dict(workload=w, N=n, cell=c, ratio=np.exp(diff), lo=np.exp(diff - q * se), hi=np.exp(diff + q * se)))
bc = pd.DataFrame(brows)
bc.to_csv(f"{OUT}/browser_contrast.csv", index=False)
for w in ["core", "full"]:
    for c in CELLS:
        parts = [f"{n//1000}k {bc.query('workload==@w and N==@n and cell==@c').ratio.iloc[0]:.2f}" for n in NS]
        p(f"- {w} {c}: " + ", ".join(parts))

# ---------------------------------------------------------------- cost per row and iteration
h("Marginal cost per row per IRLS iteration (µs), OLS of session-median computeMs on N×iterations")
itn = d.groupby("N")["iterations"].first().to_dict()
srows = []
for (b, s, w, c), g in sm.groupby(["browser", "sessionId", "workload", "cell"]):
    x = np.array([n * itn[n] for n in g["N"]], dtype=float)
    y = g["computeMs"].values
    slope, icpt = np.polyfit(x, y, 1)
    srows.append(dict(browser=b, sessionId=s, workload=w, cell=c, slope_us=slope * 1000, intercept_ms=icpt))
sl = pd.DataFrame(srows)
sls = []
for (b, w, c), g in sl.groupby(["browser", "workload", "cell"]):
    m, lo, hi, k = t_ci(g["slope_us"])
    mi, loi, hii, _ = t_ci(g["intercept_ms"])
    sls.append(dict(browser=b, workload=w, cell=c, slope=m, lo=lo, hi=hi, intercept=mi))
sls = pd.DataFrame(sls)
sls.to_csv(f"{OUT}/slopes.csv", index=False)
for r in sls.itertuples():
    p(f"- {r.browser}/{r.workload}/{r.cell}: {r.slope:.3f} µs ({r.lo:.3f}–{r.hi:.3f}), intercept {r.intercept:.2f} ms")
# native slope
nat_s = []
for inv, g in nmed.groupby("invocation"):
    x = np.array([n * itn[n] for n in g["N"]], dtype=float)
    nat_s.append(np.polyfit(x, g["computeMs"].values, 1)[0] * 1000)
m, lo, hi, k = t_ci(nat_s)
p(f"- native/core: {m:.3f} µs ({lo:.3f}–{hi:.3f})")

# ---------------------------------------------------------------- start-up
h("Start-up of per-run workers (mean of session medians, ms)")
for b in BROWSERS:
    for w in ["core", "full"]:
        for c in ["B-cold", "D-cold"]:
            g = sm.query("browser==@b and workload==@w and cell==@c")
            parts = []
            for col in ["spawnMs", "initMs", "engineInitMs", "startupMs"]:
                parts.append(f"{col} {g[col].mean():.1f}")
            p(f"- {b}/{w}/{c}: " + ", ".join(parts) + f" (range over N of startupMs {g.groupby('N')['startupMs'].mean().min():.1f}–{g.groupby('N')['startupMs'].mean().max():.1f})")

# ---------------------------------------------------------------- drift / battery
h("Drift check: session index (1–10) vs session-median responseMs, per browser/workload/N/cell")
drows = []
for (b, w, n, c), g in sm.groupby(["browser", "workload", "N", "cell"]):
    idx = g["sessionId"].str[-2:].astype(int).values
    y = np.log(g["responseMs"].values)
    rho, pv = st.spearmanr(idx, y)
    early = np.exp(y[idx <= 3].mean() - y[idx > 3].mean())
    drows.append(dict(browser=b, workload=w, N=n, cell=c, rho=rho, p=pv, early_vs_late=early))
dr = pd.DataFrame(drows)
dr.to_csv(f"{OUT}/drift.csv", index=False)
p(f"Spearman rho range {dr.rho.min():.2f} to {dr.rho.max():.2f}; cells with p<0.05: {int((dr.p < 0.05).sum())} of {len(dr)}")
p(f"Ratio sessions 1–3 vs 4–10: median {dr.early_vs_late.median():.3f}, range {dr.early_vs_late.min():.3f}–{dr.early_vs_late.max():.3f}")
cv = sm.groupby(["browser", "workload", "N", "cell"])["responseMs"].agg(lambda x: x.std(ddof=1) / x.mean()).reset_index(name="cv")
cv.to_csv(f"{OUT}/between_session_cv.csv", index=False)
p(f"Between-session CV of session medians: median {cv.cv.median()*100:.1f}%, max {cv.cv.max()*100:.1f}% "
  f"({cv.loc[cv.cv.idxmax(), ['browser','workload','N','cell']].to_dict()})")
within = d.groupby(["browser", "sessionId", "workload", "N", "cell"])["responseMs"].agg(lambda x: x.std(ddof=1) / x.mean()).reset_index(name="cv")
p(f"Within-session CV (30 runs): median {within.cv.median()*100:.1f}%")

# ---------------------------------------------------------------- memory
h("Wasm linear memory (MB), session medians, mean over sessions")
for b in BROWSERS:
    for w in ["core", "full"]:
        parts = []
        for n in NS:
            g = sm.query("browser==@b and workload==@w and N==@n and cell=='D-cold'")
            parts.append(f"{n//1000}k {g['wasmMemoryMB'].mean():.1f}")
        p(f"- {b}/{w} D-cold (per-run peak): " + ", ".join(parts))

# ---------------------------------------------------------------- accuracy
h("Accuracy")
bcols = [c for c in d.columns if c.startswith("B_")]
scols = [c for c in d.columns if c.startswith("SE_")]
for w in ["core", "full"]:
    for n in NS:
        g = d.query("workload==@w and N==@n")
        ref = g.query("cell=='D-warm' and browser=='chrome'").iloc[0]
        db = np.max(np.abs(g[bcols].values - ref[bcols].values.astype(float)))
        ds = np.max(np.abs(g[scols].values - ref[scols].values.astype(float)))
        # bitwise identical within technology and browser?
        same = []
        for b in BROWSERS:
            for tech, cells in [("js", ["A", "B-warm", "B-cold"]), ("wasm", ["C", "D-warm", "D-cold"])]:
                gg = g.query("browser==@b and cell in @cells")[bcols + scols]
                same.append(f"{b}-{tech}:{'bitwise' if (gg.nunique() == 1).all() else 'differs'}")
        cross = []
        for tech, cells in [("js", ["A", "B-warm", "B-cold"]), ("wasm", ["C", "D-warm", "D-cold"])]:
            gg = g.query("cell in @cells")[bcols + scols]
            cross.append(f"{tech} across browsers:{'bitwise' if (gg.nunique() == 1).all() else 'differs'}")
        p(f"- {w} N={n:,}: max|ΔB| {db:.1e}, max|ΔSE| {ds:.1e}; " + ", ".join(same + cross))

open(f"{OUT}/numbers.md", "w").write("# Numbers digest (auto-generated)\n" + "\n".join(lines) + "\n")
print("\n".join(lines))

### `responsiveness.py`
Responsivitas selama komputasi (frame gap terpanjang, dropped frames, long tasks), responsivitas saat start-up worker, dekomposisi overhead (dispatch, return, handler), dan uji fase frame (vsync). Menulis `out/numbers_responsiveness.md`, `responsiveness.csv`, `phase_bins.csv`, `worker_phase_runs.csv`.

In [ ]:
%%writefile analysis/responsiveness.py
"""Responsiveness, start-up responsiveness and overhead decomposition."""
import numpy as np
import pandas as pd

from common import BROWSERS, CELLS, NS, OUT, load_runs, session_medians, t_ci

d = load_runs()
lines = []
p = lines.append

p("\n## Responsiveness during the computation window\n")
d["dropped_any"] = d["droppedFrames"] > 0
d["lt_any"] = d["longTaskCount"] > 0
rows = []
for (b, w, n, c), g in d.groupby(["browser", "workload", "N", "cell"]):
    sm = g.groupby("sessionId")["maxFrameGapMs"].median()
    m, lo, hi, k = t_ci(sm)
    rows.append(dict(browser=b, workload=w, N=n, cell=c,
                     gap_mean=m, gap_lo=lo, gap_hi=hi,
                     gap_p95=g["maxFrameGapMs"].quantile(0.95), gap_max=g["maxFrameGapMs"].max(),
                     dropped_med=g.groupby("sessionId")["droppedFrames"].median().mean(),
                     share_dropped=g["dropped_any"].mean(),
                     share_longtask=g["lt_any"].mean() if g["longTaskCount"].notna().any() else np.nan,
                     lt_total_med=g.groupby("sessionId")["longTaskTotalMs"].median().mean()
                     if g["longTaskTotalMs"].notna().any() else np.nan))
r = pd.DataFrame(rows)
r.to_csv(f"{OUT}/responsiveness.csv", index=False)
for b in BROWSERS:
    for w in ["core", "full"]:
        p(f"\n**{b}/{w}**: median max frame gap (ms, mean of session medians) | share of runs with ≥1 dropped frame | share with long task")
        p("| N | " + " | ".join(CELLS) + " |")
        p("|---|" + "---|" * len(CELLS))
        for n in NS:
            vals = []
            for c in CELLS:
                x = r.query("browser==@b and workload==@w and N==@n and cell==@c").iloc[0]
                lt = "" if np.isnan(x.share_longtask) else f" / LT {x.share_longtask*100:.0f}%"
                vals.append(f"{x.gap_mean:.1f} / {x.share_dropped*100:.0f}%{lt}")
            p(f"| {n:,} | " + " | ".join(vals) + " |")

p("\nWorker cells pooled (all N, both workloads): max-gap p95 and max, share dropped")
for b in BROWSERS:
    for c in ["B-warm", "B-cold", "D-warm", "D-cold"]:
        g = d.query("browser==@b and cell==@c")
        p(f"- {b} {c}: p95 {g['maxFrameGapMs'].quantile(0.95):.1f} ms, max {g['maxFrameGapMs'].max():.1f} ms, "
          f"runs with dropped frames {int(g['dropped_any'].sum())}/{len(g)}, long tasks {int(g['lt_any'].sum()) if g['longTaskCount'].notna().any() else 'n/a'}")
p("\nMain-thread cells: share of runs that were a long task (>50 ms), Chrome")
for w in ["core", "full"]:
    for n in NS:
        parts = []
        for c in ["A", "C"]:
            g = d.query("browser=='chrome' and workload==@w and N==@n and cell==@c")
            parts.append(f"{c} {g['lt_any'].mean()*100:.0f}%")
        p(f"- {w} N={n:,}: " + ", ".join(parts))
p("\nMain-thread cells: median dropped frames (mean of session medians) at N=100,000")
for b in BROWSERS:
    for w in ["core", "full"]:
        parts = [f"{c} {r.query('browser==@b and workload==@w and N==100000 and cell==@c').dropped_med.iloc[0]:.0f}" for c in ["A", "C"]]
        p(f"- {b}/{w}: " + ", ".join(parts))

p("\n## Start-up responsiveness of per-run workers (window from new Worker to ready)\n")
for b in BROWSERS:
    for c in ["B-cold", "D-cold"]:
        g = d.query("browser==@b and cell==@c")
        lt = g["spawnLongTaskCount"]
        p(f"- {b} {c}: spawn max-gap median {g['spawnMaxFrameGapMs'].median():.1f} ms, p95 {g['spawnMaxFrameGapMs'].quantile(0.95):.1f}, max {g['spawnMaxFrameGapMs'].max():.1f}; "
          f"runs with dropped frames {int((g['spawnDroppedFrames']>0).sum())}/{len(g)}; long tasks {int((lt>0).sum()) if lt.notna().any() else 'n/a'}")

p("\n## Overhead decomposition, worker cells (median over runs, mean of session medians, ms)\n")
sm = session_medians(d, ["overheadMs", "dispatchLatencyMs", "returnLatencyMs", "workerHandlerMs", "computeMs"])
sm["handlerExtra"] = sm["workerHandlerMs"] - sm["computeMs"]
for b in BROWSERS:
    for w in ["core", "full"]:
        for c in ["B-warm", "D-warm", "B-cold", "D-cold"]:
            parts = []
            for n in NS:
                g = sm.query("browser==@b and workload==@w and N==@n and cell==@c")
                parts.append(f"{n//1000}k: ovh {g.overheadMs.mean():.2f} = disp {g.dispatchLatencyMs.mean():.2f} + ret {g.returnLatencyMs.mean():.2f} + extra {g.handlerExtra.mean():.2f}")
            p(f"- {b}/{w}/{c}: " + "; ".join(parts))

p("\n## Frame-phase test: does a result wait for a rendering step?\n")
wk = d[d["cell"].isin(["B-warm", "D-warm", "B-cold", "D-cold"])].copy()
# a frame callback ran between the worker's post and the main thread's receipt
wk["frame_between"] = wk["t1PhaseMs"] < wk["returnLatencyMs"]
wk["post_phase"] = (wk["t1PhaseMs"] - wk["returnLatencyMs"]) % 16.667
res = []
for b in BROWSERS:
    g = wk.query("browser==@b")
    a = g[g.frame_between]["returnLatencyMs"]
    nb = g[~g.frame_between]["returnLatencyMs"]
    p(f"- {b}: frame callback between post and receipt in {g.frame_between.mean()*100:.1f}% of worker runs; "
      f"return latency median {a.median():.2f} ms (p90 {a.quantile(.9):.2f}) when a frame intervened vs {nb.median():.2f} ms (p90 {nb.quantile(.9):.2f}) otherwise")
    for c in ["B-warm", "D-warm"]:
        for w in ["core"]:
            for n in NS:
                gg = g.query("cell==@c and workload==@w and N==@n")
                p(f"    · {c}/{w}/N={n//1000}k: frame-between {gg.frame_between.mean()*100:.0f}%, ret median {gg.returnLatencyMs.median():.2f}, "
                  f"overhead median {gg.overheadMs.median():.2f}")
# binned by phase of the post relative to the last frame (only runs without intervening frame are well-defined; use all with modulo)
wk["phase_bin"] = pd.cut(wk["post_phase"], bins=np.arange(0, 17.5, 1.0), include_lowest=True)
ph = wk.groupby(["browser", "phase_bin"], observed=True)["returnLatencyMs"].agg(["median", "count"]).reset_index()
ph.to_csv(f"{OUT}/phase_bins.csv", index=False)
wk[["browser", "workload", "N", "cell", "post_phase", "returnLatencyMs", "dispatchLatencyMs", "overheadMs", "frame_between", "t0PhaseMs"]].to_csv(f"{OUT}/worker_phase_runs.csv", index=False)

open(f"{OUT}/numbers_responsiveness.md", "w").write("# Responsiveness digest\n" + "\n".join(lines) + "\n")
print("\n".join(lines))

### `extra.py`
Angka tambahan: porsi run main thread dengan gap > 50 ms dan > 200 ms, serta stall > 1 s di run worker. Menulis `out/numbers_extra.md`.

In [ ]:
%%writefile analysis/extra.py
"""Extra numbers: main-thread blocking thresholds, stalls, Firefox JS cold<warm, crossings."""
import numpy as np, pandas as pd
from common import load_runs, OUT
d = load_runs()
L = ["\n## Extra numbers\n"]
m = d[d.cell.isin(["A", "C"])]
for b in ["chrome", "firefox"]:
    for w in ["core", "full"]:
        parts = []
        for n in [1000, 10000, 50000, 100000]:
            g = m.query("browser==@b and workload==@w and N==@n")
            parts.append(f"{n//1000}k: >50ms A {100*(g.query('cell==\"A\"').maxFrameGapMs>50).mean():.0f}% C {100*(g.query('cell==\"C\"').maxFrameGapMs>50).mean():.0f}%; >200ms A {100*(g.query('cell==\"A\"').maxFrameGapMs>200).mean():.0f}% C {100*(g.query('cell==\"C\"').maxFrameGapMs>200).mean():.0f}%")
        L.append(f"- {b}/{w} main-thread frame gap: " + " | ".join(parts))
wk = d[~d.cell.isin(["A", "C"])]
st = wk[wk.maxFrameGapMs > 1000]
L.append(f"\nStalls >1 s in worker runs: {len(st)} of {len(wk)} ({100*len(st)/len(wk):.2f}%); in main-thread runs gap−compute max {(m.maxFrameGapMs-m.computeMs).max():.1f} ms")
L.append(f"Stalled runs e2e range {st.e2eMs.min():.0f}–{st.e2eMs.max():.0f} ms; computeMs normal (median {st.computeMs.median():.0f})")
# worker frame gap overall
L.append(f"Worker runs: median longest frame gap {wk.maxFrameGapMs.median():.1f} ms; share >50 ms {(wk.maxFrameGapMs>50).mean()*100:.2f}%")
# Chrome vs Firefox start-up
open(f"{OUT}/numbers_extra.md", "w").write("\n".join(L) + "\n")
print("\n".join(L))

### `make_figures.py`
Gambar 1–4 naskah: jalur eksekusi, waktu respons vs N, penalti siklus hidup, dan responsivitas. PDF untuk LaTeX, PNG untuk Word, lebar 16 cm.

In [ ]:
%%writefile analysis/make_figures.py
"""Publication figures for the v2 manuscript (PDF + PNG, 16 cm wide)."""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from common import FIG, NS, OUT, load_runs

JS, WASM, NAT, INK, MUTED, GRID = "#eb6834", "#2a78d6", "#52514e", "#0b0b0b", "#8a8984", "#e6e5e1"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.edgecolor": MUTED,
    "axes.linewidth": 0.6, "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK,
    "ytick.labelcolor": INK, "axes.labelcolor": INK, "axes.titlecolor": INK, "pdf.fonttype": 42,
    "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
})
W = 16 / 2.54
STY = {
    "A": dict(color=JS, ls="-", marker="o", label="A  JS · main thread"),
    "B-warm": dict(color=JS, ls="--", marker="s", label="B-warm  JS · persistent worker"),
    "B-cold": dict(color=JS, ls=":", marker="^", label="B-cold  JS · per-run worker"),
    "C": dict(color=WASM, ls="-", marker="o", label="C  Wasm · main thread"),
    "D-warm": dict(color=WASM, ls="--", marker="s", label="D-warm  Wasm · persistent worker"),
    "D-cold": dict(color=WASM, ls=":", marker="^", label="D-cold  Wasm · per-run worker"),
}
BNAME = {"chrome": "Chrome 155", "firefox": "Firefox 158"}


def save(fig, name):
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", bbox_inches="tight")
    plt.close(fig)


def style_log_n(ax):
    ax.set_xscale("log")
    ax.set_xticks(NS)
    ax.set_xticklabels(["1k", "10k", "50k", "100k"])
    ax.minorticks_off()
    ax.grid(True, which="major", color=GRID, lw=0.5)
    ax.set_axisbelow(True)


rt = pd.read_csv(f"{OUT}/response_time.csv")
nat = pd.read_csv(f"{OUT}/native.csv")
eff = pd.read_csv(f"{OUT}/effects.csv")
resp = pd.read_csv(f"{OUT}/responsiveness.csv")

# ------------------------------------------------------------------ Figure 1: execution paths
fig, ax = plt.subplots(figsize=(W, 3.1))
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 34)
ax.axis("off")


def box(x, y, w, h, text, fc, ec=None, tc=INK, fs=5.9, ls="-"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15,rounding_size=0.6",
                                fc=fc, ec=ec or fc, lw=0.8, ls=ls))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color=tc)


def arrow(x0, y0, x1, y1):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=6, lw=0.7, color=MUTED))


def span(y, text):
    ax.annotate("", xy=(72.5, y), xytext=(34, y), arrowprops=dict(arrowstyle="<->", lw=0.6, color=INK))
    ax.text(53.25, y - 1.3, text, ha="center", va="center", fontsize=6.1, color=INK)


LIGHT_JS, LIGHT_W, LIGHT_G = "#fbe1d6", "#d8e6f8", "#efeeea"
ax.text(0, 30.0, "Main thread\n(A, C)", ha="left", va="center", fontsize=6.5, color=INK, fontweight="bold")
ax.text(0, 17.2, "Persistent worker\n(B-warm, D-warm)", ha="left", va="center", fontsize=6.5, color=INK, fontweight="bold")
ax.text(0, 4.6, "Per-run worker\n(B-cold, D-cold)", ha="left", va="center", fontsize=6.5, color=INK, fontweight="bold")
# row 1
box(21, 28, 11, 4.2, "data\npreparation", LIGHT_G)
box(34, 28, 38.5, 4.2, "IRLS core or full analysis (JS or Wasm)\nexecuted on the main thread", LIGHT_W, ec=WASM)
box(75, 28, 12, 4.2, "output tables\n(full workload)", LIGHT_G, fs=5.6)
ax.text(88, 30.1, "responseMs", fontsize=6.1, color=INK, va="center")
span(26.4, "computeMs = e2eMs; the page cannot draw frames meanwhile")
# row 2
box(20.5, 18.2, 12, 3.6, "worker created\nonce per batch", "white", ec=MUTED, ls="--", fs=5.5)
box(34, 18.2, 5.5, 3.6, "post\nX, y", LIGHT_G)
box(64.5, 18.2, 7, 3.6, "result\nmessage", LIGHT_G)
box(75, 18.2, 12, 3.6, "output tables\n(full workload)", LIGHT_G, fs=5.6)
ax.text(52, 20.0, "page keeps drawing frames", ha="center", va="center", fontsize=5.7, color=MUTED, style="italic")
box(40.5, 13.6, 23, 3.4, "computation in worker", LIGHT_W, ec=WASM)
arrow(39.5, 18.2, 42.5, 17.0)
arrow(61.5, 17.0, 65, 18.2)
ax.text(88, 20.0, "responseMs", fontsize=6.1, color=INK, va="center")
span(12.2, "e2eMs = dispatch latency + computeMs + return latency")
# row 3
box(21, 5.6, 11, 3.6, "new Worker\n+ init", LIGHT_JS, ec=JS)
box(34, 5.6, 5.5, 3.6, "post\nX, y", LIGHT_G)
box(64.5, 5.6, 7, 3.6, "result,\nterminate", LIGHT_G)
box(75, 5.6, 12, 3.6, "output tables\n(full workload)", LIGHT_G, fs=5.6)
ax.text(52, 7.4, "page keeps drawing frames", ha="center", va="center", fontsize=5.7, color=MUTED, style="italic")
box(40.5, 1.0, 23, 3.4, "computation in a new worker", LIGHT_W, ec=WASM, fs=5.6)
arrow(39.5, 5.6, 42.5, 4.4)
arrow(61.5, 4.4, 65, 5.6)
ax.text(26.5, 0.6, "startupMs\n(spawn + init + engine)", ha="center", va="center", fontsize=5.9, color=JS)
ax.text(88, 7.4, "responseMs\n(incl. startupMs)", fontsize=6.1, color=INK, va="center")
save(fig, "fig1_execution_paths")

# ------------------------------------------------------------------ Figure 2: response time vs N
fig, axes = plt.subplots(2, 2, figsize=(W, 4.6), sharex=True)
for i, w in enumerate(["core", "full"]):
    for j, b in enumerate(["chrome", "firefox"]):
        ax = axes[i, j]
        for c, s in STY.items():
            g = rt.query("browser==@b and workload==@w and cell==@c and metric=='responseMs'").sort_values("N")
            ax.errorbar(g["N"], g["mean"], yerr=[g["mean"] - g["lo"], g["hi"] - g["mean"]], color=s["color"],
                        ls=s["ls"], marker=s["marker"], ms=4, lw=1.3, capsize=0, mec="white", mew=0.6)
        if w == "core":
            ax.plot(nat["N"], nat["mean"], color=NAT, ls="-", marker="D", ms=3.5, lw=1.0, mec="white", mew=0.6)
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        style_log_n(ax)
        ax.set_title(f"{BNAME[b]} · {'IRLS core' if w == 'core' else 'full analysis'}", loc="left")
        if j == 0:
            ax.set_ylabel("response time (ms)")
        if i == 1:
            ax.set_xlabel("observations N")
_h = {k: Line2D([], [], color=s["color"], ls=s["ls"], marker=s["marker"], ms=4, lw=1.3, mec="white", label=s["label"]) for k, s in STY.items()}
_h["native"] = Line2D([], [], color=NAT, marker="D", ms=3.5, lw=1.0, mec="white", label="native Rust (core, reference)")
handles = [_h[k] for k in ["A", "C", "B-warm", "D-warm", "B-cold", "D-cold", "native"]]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.07))
fig.tight_layout()
save(fig, "fig2_response_time")

# ------------------------------------------------------------------ Figure 3: lifecycle penalty
fig, axes = plt.subplots(2, 2, figsize=(W, 4.4), sharex=True)
for i, w in enumerate(["core", "full"]):
    for j, metric in enumerate(["computeMs", "responseMs"]):
        ax = axes[i, j]
        ax.axhline(1, color=MUTED, lw=0.7)
        for tech, (cold, warm), col in [("JS", ("B-cold", "B-warm"), JS), ("Wasm", ("D-cold", "D-warm"), WASM)]:
            name = f"Cold vs warm, {tech} ({cold}/{warm})"
            for b, ls, mk in [("chrome", "-", "o"), ("firefox", "--", "s")]:
                g = eff.query("contrast==@name and metric==@metric and browser==@b and workload==@w").sort_values("N")
                ax.errorbar(g["N"], g["ratio"], yerr=[g["ratio"] - g["lo"], g["hi"] - g["ratio"]], color=col, ls=ls,
                            marker=mk, ms=4, lw=1.3, mec="white", mew=0.6, capsize=0)
        ax.set_yscale("log")
        ax.set_yticks([0.8, 1, 1.5, 2, 3, 5, 10, 15])
        ax.set_yticklabels(["0.8", "1", "1.5", "2", "3", "5", "10", "15"])
        ax.minorticks_off()
        style_log_n(ax)
        ax.set_title(("(a) " if (i, j) == (0, 0) else "(b) " if (i, j) == (0, 1) else "(c) " if (i, j) == (1, 0) else "(d) ")
                     + ("computation only" if metric == "computeMs" else "response time incl. start-up")
                     + (" · IRLS core" if w == "core" else " · full analysis"), loc="left")
        if j == 0:
            ax.set_ylabel("per-run / persistent")
        if i == 1:
            ax.set_xlabel("observations N")
handles = [Line2D([], [], color=JS, ls="-", marker="o", ms=4, lw=1.3, mec="white", label="JS · Chrome"),
           Line2D([], [], color=JS, ls="--", marker="s", ms=4, lw=1.3, mec="white", label="JS · Firefox"),
           Line2D([], [], color=WASM, ls="-", marker="o", ms=4, lw=1.3, mec="white", label="Wasm · Chrome"),
           Line2D([], [], color=WASM, ls="--", marker="s", ms=4, lw=1.3, mec="white", label="Wasm · Firefox")]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()
save(fig, "fig3_lifecycle")

# ------------------------------------------------------------------ Figure 4: responsiveness
d = load_runs()
wk = pd.read_csv(f"{OUT}/worker_phase_runs.csv")
fig = plt.figure(figsize=(W, 2.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.9], wspace=0.42)
for j, b in enumerate(["chrome", "firefox"]):
    ax = fig.add_subplot(gs[0, j])
    ax.axhline(16.7, color=MUTED, lw=0.6, ls="-")
    ax.axhline(50, color=MUTED, lw=0.6, ls="--")
    ax.text(1000, 54, "50 ms (long task)", fontsize=5.8, color=MUTED, va="bottom")
    ax.text(1000, 15.2, "16.7 ms (one frame)", fontsize=5.8, color=MUTED, va="top")
    for c, s in STY.items():
        g = resp.query("browser==@b and workload=='full' and cell==@c").sort_values("N")
        ax.plot(g["N"], g["gap_mean"], color=s["color"], ls=s["ls"], marker=s["marker"], ms=4, lw=1.3, mec="white", mew=0.6)
    ax.set_yscale("log")
    ax.set_yticks([15, 20, 50, 100, 200, 300])
    ax.set_yticklabels(["15", "20", "50", "100", "200", "300"])
    ax.minorticks_off()
    style_log_n(ax)
    ax.set_xticklabels(["1k", "10k", "", "100k"])
    ax.set_ylim(12, 360)
    ax.set_title(f"({'a' if j == 0 else 'b'}) {BNAME[b]}", loc="left")
    ax.set_xlabel("observations N")
    if j == 0:
        ax.set_ylabel("longest frame gap (ms)")
ax = fig.add_subplot(gs[0, 2])
data, labels, cols = [], [], []
for b in ["chrome", "firefox"]:
    g = wk.query("browser==@b")
    for fb, lab in [(False, "no frame"), (True, "frame")]:
        data.append(g[g.frame_between == fb]["returnLatencyMs"].values)
        share = (g.frame_between == fb).mean() * 100
        labels.append(f"{'none' if not fb else 'frame'}\n{share:.0f}%")
        cols.append(WASM if fb else MUTED)
bp = ax.boxplot(data, widths=0.55, showfliers=False, patch_artist=True, medianprops=dict(color=INK, lw=1.0),
                whiskerprops=dict(color=MUTED, lw=0.7), capprops=dict(color=MUTED, lw=0.7))
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.35)
    patch.set_edgecolor(c)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(labels, fontsize=5.8)
ax.text(1.5, -0.21, "Chrome", transform=ax.get_xaxis_transform(), ha="center", fontsize=6.3, color=INK)
ax.text(3.5, -0.21, "Firefox", transform=ax.get_xaxis_transform(), ha="center", fontsize=6.3, color=INK)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_ylabel("return latency (ms)")
ax.set_title("(c) worker → main delay", loc="left")
ax.grid(True, axis="y", color=GRID, lw=0.5)
ax.set_axisbelow(True)
handles = [Line2D([], [], color=s["color"], ls=s["ls"], marker=s["marker"], ms=4, lw=1.3, mec="white", label=k) for k, s in STY.items()]
fig.legend(handles=handles, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.36, 1.08))
save(fig, "fig4_responsiveness")
print("figures written")

### `make_tables.py`
Isi Tabel 2 (waktu respons) dan Tabel 3 (rasio efek) dalam format LaTeX booktabs, ditulis ke `tables/`.

In [ ]:
%%writefile analysis/make_tables.py
"""LaTeX table bodies (booktabs) for Tables 2 and 3 of the v2 manuscript."""
import numpy as np
import pandas as pd

from common import CELLS, NS, OUT, ROOT

rt = pd.read_csv(f"{OUT}/response_time.csv")
nat = pd.read_csv(f"{OUT}/native.csv")
eff = pd.read_csv(f"{OUT}/effects.csv")
os_tab = f"{ROOT}/tables"
import os

os.makedirs(os_tab, exist_ok=True)


def f1(x):
    return f"{x:.1f}" if x < 100 else f"{x:.0f}"


# ---------------- Table 2: response time
r = rt.query("metric=='responseMs'").copy()
r["half_rel"] = (r["hi"] - r["lo"]) / 2 / r["mean"]
maxhw = r["half_rel"].max() * 100
lines = []
for w, title in [("core", "IRLS core workload"), ("full", "Full analysis workload")]:
    lines.append(r"\midrule" if w == "full" else "")
    lines.append(rf"\multicolumn{{9}}{{@{{}}l}}{{\textit{{{title}}}}}\\")
    for b, bn in [("chrome", "Chrome"), ("firefox", "Firefox")]:
        for i, n in enumerate(NS):
            cells = [f1(r.query("browser==@b and workload==@w and N==@n and cell==@c")["mean"].iloc[0]) for c in CELLS]
            nv = f1(nat.query("N==@n")["mean"].iloc[0]) if w == "core" else "--"
            lab = bn if i == 0 else ""
            lines.append(f"{lab} & {n:,} & " + " & ".join(cells) + f" & {nv} \\\\")
        if b == "chrome":
            lines.append(r"\addlinespace")
body = "\n".join(l for l in lines if l)
open(f"{os_tab}/table2_response_body.tex", "w").write(body + "\n")
print(f"Table 2: max relative 95% CI half-width {maxhw:.1f}%")
w95 = r.sort_values("half_rel", ascending=False).head(3)[["browser", "workload", "N", "cell", "half_rel"]]
print(w95)

# ---------------- Table 3: effects (ratios of response time)
order = [
    ("Technology: Wasm / JS (main thread, C/A)", "Wasm vs JS, main (C/A)"),
    ("Technology: Wasm / JS (persistent worker, D/B)", "Wasm vs JS, worker warm (D-warm/B-warm)"),
    ("Technology: Wasm / JS (per-run worker, D/B)", "Wasm vs JS, worker cold (D-cold/B-cold)"),
    ("Location: worker / main (JS, B-warm/A)", "Worker vs main, JS warm (B-warm/A)"),
    ("Location: worker / main (Wasm, D-warm/C)", "Worker vs main, Wasm warm (D-warm/C)"),
    ("Lifecycle: per-run / persistent (JS)", "Cold vs warm, JS (B-cold/B-warm)"),
    ("Lifecycle: per-run / persistent (Wasm)", "Cold vs warm, Wasm (D-cold/D-warm)"),
]
SEL = [1000, 10000, 100000]
lines = []
for label, key in order:
    lines.append(rf"\multicolumn{{7}}{{@{{}}l}}{{\textit{{{label}}}}}\\")
    for b, bn in [("chrome", "Chrome"), ("firefox", "Firefox")]:
        vals = []
        for w in ["core", "full"]:
            for n in SEL:
                x = eff.query("contrast==@key and metric=='responseMs' and browser==@b and workload==@w and N==@n").iloc[0]
                vals.append(f"{x.ratio:.2f} ({x.lo:.2f}--{x.hi:.2f})")
        lines.append(rf"\quad {bn} & " + " & ".join(vals) + r" \\")
open(f"{os_tab}/table3_effects_body.tex", "w").write("\n".join(lines) + "\n")
print("tables written")

## 3. Analisis
### 3.1 `analyze.py`

In [ ]:
jalankan([sys.executable, "analyze.py"], cwd="analysis")
display(Markdown(open("out/numbers.md", encoding="utf-8").read()))

### 3.2 `responsiveness.py` dan `extra.py`

In [ ]:
jalankan([sys.executable, "responsiveness.py"], cwd="analysis")
jalankan([sys.executable, "extra.py"], cwd="analysis")
display(Markdown(open("out/numbers_responsiveness.md", encoding="utf-8").read()))
display(Markdown(open("out/numbers_extra.md", encoding="utf-8").read()))

### 3.3 Melihat CSV
Dua ringkasan cepat dari `out/`: waktu respons (rata-rata median sesi, ms) dan rasio efek dengan CI 95%. Semua CSV lain ada di folder `out/` (panel **Files** di kiri).

In [ ]:
rt = pd.read_csv("out/response_time.csv")
waktu = (rt[rt.metric == "responseMs"]
         .pivot(index=["workload", "browser", "N"], columns="cell", values="mean")
         [["A", "B-warm", "B-cold", "C", "D-warm", "D-cold"]].round(1))
display(waktu)

eff = pd.read_csv("out/effects.csv")
e = eff[eff.metric == "responseMs"].copy()
e["rasio (CI 95%)"] = [f"{r:.2f} ({lo:.2f}–{hi:.2f})" for r, lo, hi in zip(e.ratio, e.lo, e.hi)]
efek = (e.pivot(index=["contrast", "browser"], columns=["workload", "N"], values="rasio (CI 95%)")
        .reindex(eff.contrast.unique(), level=0))
display(efek)

## 4. Gambar dan tabel

In [ ]:
jalankan([sys.executable, "make_figures.py"], cwd="analysis")
for f in sorted(glob.glob("figures/fig*.png")):
    display(Markdown(f"**{f}**"))
    display(Image(f, width=900))

In [ ]:
jalankan([sys.executable, "make_tables.py"], cwd="analysis")
for f in sorted(glob.glob("tables/*.tex")):
    print(f"===== {f}")
    print(open(f, encoding="utf-8").read())

## 5. Unduh hasil
Zip berisi `out/`, `figures/`, `tables/`, dan `analysis/`. Di Colab, unduhan dimulai otomatis.

In [ ]:
ISI = ["out", "figures", "tables", "analysis"]

zip_keluar = os.path.join(os.path.dirname(os.path.abspath(WORKDIR)), "analysis.zip")
with zipfile.ZipFile(zip_keluar, "w", zipfile.ZIP_DEFLATED) as z:
    for item in ISI:
        for f in sorted(glob.glob(f"{item}/**/*", recursive=True)):
            if os.path.isfile(f) and "__pycache__" not in f:
                z.write(f, os.path.join("paper_v2", f))
    n = len(z.namelist())
print(f"{zip_keluar}: {n} berkas, {os.path.getsize(zip_keluar) / 1e6:.1f} MB")

if SALIN_KE_DRIVE:
    os.makedirs(SALIN_KE_DRIVE, exist_ok=True)
    shutil.copy(zip_keluar, SALIN_KE_DRIVE)
    print("Disalin ke", SALIN_KE_DRIVE)
if DI_COLAB:
    from google.colab import files
    files.download(zip_keluar)